
# ✅ Core Vision Perfect V1 — 03 · Test the System & Trained Model

Sanity-checks the full retrieval stack on Colab: loads the engine, runs
Vietnamese queries end-to-end (KIS / TRAKE / AVS), measures latency, writes a
sample submission CSV, validates + packages it Codabench-style, optionally
scores it against a local ground truth (official formulas) and dry-runs the
automatic track. The Streamlit UI can be served via a tunnel at the end.

Prerequisite: notebook 01 (and optionally 02 for the fine-tuned tower).


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  1 · PARAMS — the ONLY cell you may need to edit                 ║
# ╚══════════════════════════════════════════════════════════════════╝
DRIVE_PROJECT_DIR = "AIC2025"        # MyDrive/<this>/{data, artifacts}
FIRST_TIME_SETUP = False             # True CHỈ cho lần ĐẦU TIÊN tạo dự án trên
#   Drive trống. Mặc định False: mount "lười metadata" sẽ KHÔNG BAO GIỜ được
#   tự đẻ thư mục dự án sinh đôi nữa (round-66 — bài học 07 live).
REPO_URL  = "https://github.com/ledinhminhquan/Core-Vision_Perfect_V1.git"
REPO_REF  = "main"

# Which dense encoders to build indexes for (order = ensemble order).
#   "siglip2"          multilingual default (needed for training too)
#   "openclip"         English lane (DFN5B ViT-H/14-378) — strongest with translation
#   "qwen_embed"       optional HEAVY lane (Qwen embedding tower — strong, slow)
#   "provided_clip32"  organiser features — instant, no GPU (L-batches only);
#                      auto-added in the catalog cell when clip-features-32 exists
EMBED_MODELS = ["siglip2", "openclip"]

# Copy keyframes from Drive → local disk before embedding (much faster I/O).
COPY_KEYFRAMES_LOCAL = True

# Aux indexes to build (each is resumable; captions are the slowest).
RUN_OCR, RUN_ASR, RUN_CAPTIONS = True, True, True
CAPTION_STRIDE = 4                   # caption mỗi keyframe thứ 4 (round-16: đủ dày
#   cho kênh recall BM25 mà nhanh gấp đôi stride 2. NÂNG stride luôn an toàn với
#   resume: video đã caption ở stride nhỏ hơn vẫn được tính là XONG ở stride lớn
#   hơn. Mọi phiên chạy song song PHẢI dùng CÙNG một stride.

# K-batch shot detection: install TransNetV2 (the winning-team detector) for
# keyframe self-extraction. Installed --no-deps (Colab torch is never touched);
# without it extraction falls back to PySceneDetect automatically. (nb01 only)
INSTALL_TRANSNETV2 = True

# Force-rebuild toggles — mặc định False = resume/skip khi artifact đã có.
FORCE_CATALOG    = False             # rebuild the catalog parquet
FORCE_EMBED      = False             # re-embed every keyframe
FORCE_INDEX      = False             # rebuild the FAISS indexes
FORCE_AUX        = False             # redo OCR/ASR/captions from scratch
FORCE_TEXT_INDEX = False             # rebuild the persisted BM25 text index

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("params ok")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  2 · Mount Drive + folder layout + preflight write test          ║
# ╚══════════════════════════════════════════════════════════════════╝
import os, shutil, subprocess, time
from pathlib import Path

from google.colab import drive

_MP = "/content/drive"

def _drive_alive() -> bool:
    try:
        return Path(_MP, "MyDrive").exists()
    except OSError:
        return False

def _ensure_drive():
    """Mount / HỒI SINH Drive FUSE — dùng ở mọi cell dài hơi phía sau.

    Round-19 (live run 8): daemon DriveFS chết để lại mountpoint 'bẩn' →
    drive.mount kêu 'Mountpoint must not already contain files' và cả
    force_remount cũng bó tay. Trình tự cứu đúng: (1) fusermount -uz gỡ
    mount chết; (2) CHỈ khi chắc chắn không còn mount (os.path.ismount ==
    False — lúc này các entry trong mountpoint là RÁC LOCAL trên đĩa VM,
    không phải Drive thật) mới dọn sạch chúng; (3) mount lại.
    """
    for _try in range(4):
        if _drive_alive():
            return
        if _try:
            print(f"⚠ Drive FUSE chưa sống — hồi sinh (lần {_try}/3) ...")
        try:
            if os.path.ismount(_MP):
                subprocess.run(["fusermount", "-uz", _MP], capture_output=True)
                time.sleep(2)
            if os.path.isdir(_MP) and not os.path.ismount(_MP):
                for _c in os.listdir(_MP):     # rác local — KHÔNG phải Drive
                    _p = os.path.join(_MP, _c)
                    shutil.rmtree(_p, ignore_errors=True) if os.path.isdir(_p) \
                        else os.unlink(_p)
            drive.mount(_MP, force_remount=bool(_try))
        except Exception as _e:  # noqa: BLE001 — thử tiếp vòng sau
            print("   mount lỗi:", _e)
            time.sleep(5)
    if not _drive_alive():
        raise RuntimeError(
            "Không mount được Google Drive sau 4 lần thử — Runtime ▸ "
            "Disconnect and delete runtime rồi Run all lại (tiến độ đã lưu "
            "trên Drive còn nguyên).")

_ensure_drive()
assert Path("/content/drive/MyDrive").exists(), "Drive mount failed — rerun this cell"

PROJECT   = Path("/content/drive/MyDrive") / DRIVE_PROJECT_DIR
DATA_DIR  = PROJECT / "data"           # organiser dataset (merged packages)
ARTIFACTS = PROJECT / "artifacts"      # everything we build → survives disconnects
# Round-63 (live 07): mkdir NGAY trên mount còn "lười metadata" từng ĐẺ RA một
# AIC2025 SINH ĐÔI rỗng (Drive cho phép trùng tên) — từ đó mỗi phiên mới bind
# ngẫu nhiên vào bản thật hay bản rỗng và "không thấy data". Dự án đã tồn tại
# thì KHÔNG BAO GIỜ mkdir; chỉ khi chờ 3 phút vẫn không thấy (lần setup đầu
# tiên trong đời) mới được tạo.
_t0p = time.time()
while not PROJECT.exists() and time.time() - _t0p < 180:
    print(f"⏳ chưa thấy MyDrive/{DRIVE_PROJECT_DIR} — đợi metadata "
          f"({int(time.time() - _t0p)}s; TUYỆT ĐỐI không tự tạo vội) ...")
    time.sleep(10)
    try:
        list(Path("/content/drive/MyDrive").iterdir())   # cú hích ép nạp metadata
    except OSError:
        pass
if not PROJECT.exists():
    # round-66: KHÔNG BAO GIỜ tự tạo khi chưa được phép — chính là cỗ máy đẻ
    # thư mục dự án sinh đôi. Lần setup đầu tiên THẬT thì bật cờ ở cell 1.
    if not FIRST_TIME_SETUP:
        raise RuntimeError(
            f"3 phút vẫn không thấy MyDrive/{DRIVE_PROJECT_DIR} — máy ảo này "
            "hỏng metadata Drive. Runtime ▸ Disconnect and delete runtime rồi "
            "Run all lại máy mới (dữ liệu trên Drive vẫn nguyên vẹn). Nếu đây "
            "THẬT SỰ là lần đầu tạo dự án: đặt FIRST_TIME_SETUP = True ở cell 1.")
    print(f"⚠ FIRST_TIME_SETUP=True — tạo mới MyDrive/{DRIVE_PROJECT_DIR}.")
# Round-69 (audit): gate chống-sinh-đôi phải phủ cả THƯ MỤC CON — mount thấy
# AIC2025 nhưng chưa nạp children mà mkdir ngay thì data/artifacts sinh đôi
# y hệt vụ round-63, chỉ là một tầng sâu hơn.
for p in (DATA_DIR, ARTIFACTS):
    if p.exists() or FIRST_TIME_SETUP:
        p.mkdir(parents=True, exist_ok=True)
        continue
    _t0c = time.time()
    while not p.exists() and time.time() - _t0c < 120:
        print(f"⏳ chưa thấy {p.name}/ trong dự án — đợi metadata ({int(time.time() - _t0c)}s) ...")
        time.sleep(10)
        try:
            list(PROJECT.iterdir())                  # cú hích ép nạp children
        except OSError:
            pass
    if not p.exists():
        raise RuntimeError(
            f"2 phút không thấy {p.name}/ trong MyDrive/{DRIVE_PROJECT_DIR} — máy "
            "ảo hỏng metadata Drive. Runtime ▸ Disconnect and delete runtime rồi "
            "chạy máy mới; nếu đây là lần setup đầu tiên: FIRST_TIME_SETUP=True.")

# PREFLIGHT (v12): Drive PHẢI ghi/đọc được — quota đầy hay mất quyền thì
# dừng NGAY tại đây thay vì hỏng giữa chừng sau 2 giờ chạy.
_probe = ARTIFACTS / f"_write_test_{int(time.time())}.tmp"
try:
    _probe.write_text("ok", encoding="utf-8")
    assert _probe.read_text(encoding="utf-8") == "ok"
    _probe.unlink()
    print("✅ Drive write test: OK")
except Exception as e:
    raise RuntimeError(
        f"❌ Không ghi được vào Drive ({ARTIFACTS}): {e!r}\n"
        "Kiểm tra dung lượng (quota) Google Drive và quyền truy cập thư mục, "
        "rồi chạy lại ô này."
    ) from e

# DATA-PRESENCE GATE (round-20, live run 9): trên VM mới, DriveFS có thể liệt
# kê data/ ra RỖNG suốt vài phút đầu (metadata sync lười) — mkdir exist_ok ở
# trên còn CHE mất triệu chứng, để cell 7 chết khó hiểu với "Keyframes folder
# not found". Poll tới 3 phút (mỗi listdir là một cú hích ép DriveFS fetch);
# hết kiên nhẫn thì dừng TO với chẩn đoán rõ ràng.
_t0 = time.time()
_data_ok = False
while time.time() - _t0 < 180:
    try:
        if any(DATA_DIR.iterdir()):
            _data_ok = True
            break
    except OSError:
        pass
    print(f"⏳ data/ đang rỗng — đợi DriveFS sync metadata ({int(time.time() - _t0)}s) ...")
    time.sleep(10)
if not _data_ok:
    raise RuntimeError(
        "data/ trên Drive vẫn RỖNG sau 3 phút chờ. Ba nguyên nhân thường gặp:\n"
        "  1) Phiên Colab đăng nhập NHẦM tài khoản Google (kiểm tra avatar góc "
        f"phải trên) — phải là tài khoản có MyDrive/{DRIVE_PROJECT_DIR}/data;\n"
        "  2) DriveFS sync quá chậm — Runtime ▸ Disconnect and delete runtime "
        "rồi Run all lại trên máy mới;\n"
        "  3) Lần chạy đầu tiên mà chưa upload dữ liệu — ném các zip của BTC "
        f"vào MyDrive/{DRIVE_PROJECT_DIR}/data trước (docs/DRIVE_SETUP.md).\n"
        "KHÔNG có gì bị mất — dữ liệu vẫn nằm nguyên trên Drive của tài khoản đúng.")
print(f"✅ data/ nhìn thấy dữ liệu sau {int(time.time() - _t0)}s")

# HF + pip caches on Drive → models/wheels download once, not per session.
os.environ["HF_HOME"] = str(ARTIFACTS / "hf_cache")
os.environ["PIP_CACHE_DIR"] = str(ARTIFACTS / "pip_cache")
for _d in (os.environ["HF_HOME"], os.environ["PIP_CACHE_DIR"]):
    Path(_d).mkdir(parents=True, exist_ok=True)

import shutil
free_gb = shutil.disk_usage(str(PROJECT)).free / 1e9
print(f"Project: {PROJECT}")
print(f"Drive free space: {free_gb:.0f} GB")
if free_gb < 20:
    print("⚠ Less than 20 GB free on Drive — embeddings/checkpoints may not fit!")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  3 · Get repo + install dependencies (v12 discipline)            ║
# ╚══════════════════════════════════════════════════════════════════╝
# Quy tắc (học từ notebook Toxicity v12):
#   * check version qua importlib.metadata — KHÔNG import package trước khi
#     nâng cấp (import sớm sẽ ghim version cũ vào sys.modules);
#   * KHÔNG BAO GIỜ đụng torch/torchvision/torchaudio của Colab;
#   * chỉ cài đúng những gói thiếu/sai version (--prefer-binary);
#   * sau khi cài: `pip check` + micro-fix (tối đa 2 vòng, không crash),
#     rồi purge sys.modules TRƯỚC khi import cvp.
FORCE_REINSTALL_DEPS = False

import re, subprocess, sys
from pathlib import Path

REPO_DIR = Path("/content/Core-Vision_Perfect_V1")

def _run(cmd, show=None, **kw):
    # `show` masks credentials in the echoed command — a PAT-carrying clone
    # URL must NEVER be printed into the saved notebook output.
    print("$", " ".join(map(str, show or cmd)))
    return subprocess.run([str(c) for c in cmd], check=False, **kw).returncode

def _pip(args):
    return _run([sys.executable, "-m", "pip", *args])

# Private repo? Add a fine-grained PAT as Colab secret "GITHUB_TOKEN"
# (Contents: Read-only on this repo) and ENABLE its notebook-access toggle.
clone_url, _tok = REPO_URL, None
try:
    from google.colab import userdata
    _tok = userdata.get("GITHUB_TOKEN")
except Exception as _e:
    print(f"⚠ KHÔNG đọc được secret GITHUB_TOKEN ({type(_e).__name__}) — repo "
          "private sẽ KHÔNG clone được. Kiểm tra: 🔑 panel có secret tên đúng "
          "y hệt GITHUB_TOKEN và công tắc 'Notebook access' đã BẬT chưa?")
if _tok and clone_url.startswith("https://github.com/"):
    clone_url = clone_url.replace("https://", f"https://{_tok}@")
    print(f"GITHUB_TOKEN: loaded ({len(_tok)} chars, {_tok[:11]}…)")
elif not _tok:
    print("⚠ GITHUB_TOKEN trống/vắng mặt — thử clone KHÔNG xác thực "
          "(chắc chắn fail nếu repo private).")

if REPO_DIR.exists():
    _run(["git", "-C", REPO_DIR, "fetch", "--all", "-q"])
    _run(["git", "-C", REPO_DIR, "checkout", REPO_REF, "-q"])
    _run(["git", "-C", REPO_DIR, "pull", "-q"])
else:
    rc = _run(["git", "clone", "--branch", REPO_REF, clone_url, REPO_DIR],
              show=["git", "clone", "--branch", REPO_REF, REPO_URL, REPO_DIR])
    if rc != 0:  # private repo / no network → fall back to a Drive copy
        print("⚠ Clone THẤT BẠI. Nguyên nhân thường gặp, theo thứ tự:\n"
              "  1) Secret GITHUB_TOKEN sai tên / chưa bật Notebook access "
              "(xem cảnh báo phía trên);\n"
              "  2) PAT sai/hết hạn/thiếu quyền — cần fine-grained PAT với "
              "Contents: Read-only cấp cho ĐÚNG repo này;\n"
              "  3) Mạng Colab trục trặc tạm thời — chạy lại cell.")
        drive_copy = Path("/content/drive/MyDrive") / DRIVE_PROJECT_DIR / "Core-Vision_Perfect_V1"
        assert drive_copy.exists(), (
            "Clone failed and no Drive copy found. Either make the GitHub repo "
            f"reachable or upload the repo folder to {drive_copy}"
        )
        import shutil as _sh
        _sh.copytree(drive_copy, REPO_DIR)
        print("Using repo copy from Drive")

try:
    from packaging.requirements import Requirement
except ImportError:
    _pip(["install", "-q", "packaging"])
    from packaging.requirements import Requirement
from importlib.metadata import PackageNotFoundError
from importlib.metadata import version as _meta_version

# Parse requirements-colab.txt; strip any torch* line (Colab rule #1: the
# preinstalled torch/torchvision/torchaudio build must never be touched).
reqs = []
for _line in (REPO_DIR / "requirements-colab.txt").read_text(encoding="utf-8").splitlines():
    _line = _line.split("#", 1)[0].strip()
    if not _line:
        continue
    try:
        _r = Requirement(_line)
    except Exception:
        print("⚠ bỏ qua requirement không parse được:", _line)
        continue
    if _r.name.lower().replace("-", "_").startswith("torch"):
        print("skip (never touch Colab torch):", _line)
        continue
    reqs.append(_r)

def _satisfied(r):
    """Installed + in range — via importlib.metadata, WITHOUT importing it."""
    try:
        v = _meta_version(r.name)
    except PackageNotFoundError:
        return False
    return (not r.specifier) or r.specifier.contains(v, prereleases=True)

missing = [r for r in reqs if FORCE_REINSTALL_DEPS or not _satisfied(r)]
did_install = bool(missing)
if missing:
    print(f"installing {len(missing)} package(s):", ", ".join(r.name for r in missing))
    _pip(["install", "-q", "--prefer-binary", *[str(r) for r in missing]])
else:
    print("dependencies satisfied — no pip install needed")

_pip(["install", "-q", "-e", str(REPO_DIR), "--no-deps"])

# faiss: gpu wheel with cpu fallback (metadata check — no import)
def _installed(*names):
    for n in names:
        try:
            _meta_version(n)
            return n
        except PackageNotFoundError:
            pass
    return None

if _installed("faiss-gpu-cu12", "faiss-gpu", "faiss-cpu", "faiss") is None:
    if _pip(["install", "-q", "faiss-gpu-cu12"]) != 0:
        _pip(["install", "-q", "faiss-cpu"])
    did_install = True

# `pip check` + micro-fixes for known conflicts (max 2 rounds, then warn)
def _pip_check():
    r = subprocess.run([sys.executable, "-m", "pip", "check"],
                       capture_output=True, text=True)
    return r.returncode, ((r.stdout or "") + "\n" + (r.stderr or "")).strip()

if did_install:
    rc, out = _pip_check()
    for _round in (1, 2):
        if rc == 0:
            break
        # pip's two REAL formats (round-3 fix L-R3-8 — the old regex missed the
        # version-conflict wording so that repair branch never ran):
        #   "pkgA 1.0 requires pkgB, which is not installed."
        #   "pkgA 1.0 has requirement pkgB<2,>=1, but you have pkgB 3.0."
        _specs = sorted({
            m.strip()
            for m in re.findall(
                r"(?:requires|has requirement) (.+?), (?:but you have|which is not installed)", out)
            if not m.strip().lower().startswith("torch")
        })
        if not _specs:
            break
        print(f"pip check micro-fix (round {_round}):", ", ".join(_specs))
        _pip(["install", "-q", "--prefer-binary", *_specs])
        rc, out = _pip_check()
    print("pip check: OK" if rc == 0 else f"⚠ pip check còn cảnh báo (không chặn):\n{out}")

# Purge stale sys.modules of upgraded packages BEFORE importing cvp (v12).
# ONLY the packages actually (re)installed THIS run (round-11): purging every
# requirement dropped numpy/pandas from sys.modules while torch still held
# references to the old modules — the "NumPy module was reloaded" warning.
if did_install:
    _ALIAS = {"pillow": "pil", "pyyaml": "yaml", "opencv_python_headless": "cv2",
              "open_clip_torch": "open_clip", "scikit_learn": "sklearn"}
    _roots = {r.name.lower().replace("-", "_") for r in missing} | {"cvp", "faiss"}
    _roots |= {_ALIAS[n] for n in _roots & set(_ALIAS)}
    _purged = [m for m in list(sys.modules)
               if m.split(".", 1)[0].lower().replace("-", "_") in _roots]
    for _m in _purged:
        sys.modules.pop(_m, None)
    if _purged:
        print(f"purged {len(_purged)} stale sys.modules entries")

    # Sanity (round-11): the HF stack must import cleanly in a FRESH
    # interpreter — a broken hub/accelerate pairing must surface HERE with an
    # actionable message, not 5 cells later as a cryptic circular import.
    _rc = _run([sys.executable, "-c", "import transformers, accelerate"])
    if _rc != 0:
        print("⚠ transformers/accelerate KHÔNG import được — thường do phiên cài "
              "này đã hạ cấp huggingface-hub dưới mức accelerate cần. Cách sửa "
              "sạch nhất: Runtime ▸ Disconnect and delete runtime, rồi Run all "
              "lại từ đầu (mọi tiến độ đã nằm trên Drive, không mất gì).")

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
import cvp
print("cvp", cvp.__version__, "ready")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  4 · Point cvp at the data + GPU setup (TF32 / SDPA / bf16)      ║
# ╚══════════════════════════════════════════════════════════════════╝
import os, torch

os.environ["CVP_PATHS__DATA_ROOT"]      = str(DATA_DIR)
os.environ["CVP_PATHS__ARTIFACTS_ROOT"] = str(ARTIFACTS)
os.environ["CVP_SETTINGS"] = str(REPO_DIR / "configs" / "settings.yaml")

print("torch", torch.__version__, "| CUDA build", torch.version.cuda)
print("GPU available:", torch.cuda.is_available())
GPU_NAME, VRAM_GB, USE_BF16 = "cpu", 0.0, False
if torch.cuda.is_available():
    GPU_NAME = torch.cuda.get_device_name(0)
    VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
    USE_BF16 = torch.cuda.is_bf16_supported()
    # TF32 fast paths (new API with old fallback)
    try:
        torch.backends.cuda.matmul.fp32_precision = "tf32"
        torch.backends.cudnn.conv.fp32_precision = "tf32"
    except Exception:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
    for fn in ("enable_flash_sdp", "enable_mem_efficient_sdp"):
        if hasattr(torch.backends.cuda, fn):
            getattr(torch.backends.cuda, fn)(True)
print(f"GPU: {GPU_NAME} | VRAM {VRAM_GB:.0f} GB | bf16={USE_BF16}")

# Colab secrets → env (optional: Gemini query enhancement/VQA, HF pushes).
# GOOGLE_API_KEY is the name Colab's built-in "Gemini API key ▸ Import from
# Google AI Studio" button creates — the engine accepts either spelling.
try:
    from google.colab import userdata
    for _sec in ("GEMINI_API_KEY", "GOOGLE_API_KEY", "HF_TOKEN"):
        try:
            _v = userdata.get(_sec)
            if _v:
                os.environ[_sec] = _v
                print(f"secret {_sec}: loaded")
        except Exception:
            pass
except ImportError:
    pass

from cvp.config import load_settings
from cvp.utils.logging import setup_logging
settings = load_settings()
setup_logging("INFO")
print("data_root      =", settings.paths.data_root)
print("artifacts_root =", settings.paths.artifacts_root)

In [ ]:
# ── 6 · Materialize data → local disk TỪ ZIP GỐC (nhanh + miễn nhiễm FUSE) ──
# Round-14 (live-run 5): copytree 177k JPG lẻ qua Drive FUSE mất 3h+ rồi làm
# SẬP luôn cả mount ([Errno 107] Transport endpoint is not connected — mọi
# file sau đó đọc ra ENOENT). Chiến lược mới: copy CÁC FILE ZIP về local
# (ít file, to, đọc tuần tự — đúng kiểu I/O FUSE làm tốt) rồi giải nén tại
# chỗ — nhanh hơn nhiều lần, resume theo TỪNG zip, tự remount khi FUSE chết.
# Nội dung chỉ-có-trên-Drive (keyframes K-batch tự cắt, csv tái dựng…) được
# merge bù ở pha 2. Embedding/OCR/caption đọc 177k JPG từ local như cũ.
import re as _re
import shutil, time, zipfile
from pathlib import Path

# _ensure_drive/_drive_alive: bản HARDENED định nghĩa ở Ô 2 (round-19) —
# biết gỡ mount chết (fusermount -uz) + dọn mountpoint bẩn trước khi mount lại.
_ensure_drive()          # verify-R14: gate dưới stat qua FUSE — mount phải sống

def _kf_visible() -> bool:
    """DriveFS trên VM mới có thể thấy data/ nhưng CHƯA thấy subdir keyframes/
    (round-28, live nb03 run 5: gate này rơi nhầm sang nhánh Drive-direct rồi
    chết ở catalog). listdir cha = cú hích ép nạp metadata; zip Keyframes*
    cũng được chấp nhận — materialize vốn bung từ zip, không cần dir Drive."""
    try:
        list(DATA_DIR.iterdir())
        if (DATA_DIR / "keyframes").exists():
            return True
        _zn = [p.name.lower().replace("_", "-").replace(" ", "-")
               for p in DATA_DIR.glob("*.zip")]
        return any(n.startswith(("keyframes", "keyframe", "key-frames")) for n in _zn)
    except OSError:
        return False

_kf_ok = False
if COPY_KEYFRAMES_LOCAL:
    for _w in range(30):                       # tới 5 phút (round-62: VM "lười
        if _kf_visible():                      # metadata" từng cần hơn 2 phút)
            _kf_ok = True
            break
        print(f"⏳ DriveFS chưa thấy keyframes/ hay Keyframes*.zip — đợi ({_w * 10}s) ...")
        time.sleep(10)
    if not _kf_ok:
        print("⚠ 5 phút vẫn không thấy keyframes/ lẫn zip nguồn — máy ảo này dính "
              "DriveFS hỏng metadata. KHUYÊN MẠNH: Runtime ▸ Disconnect and "
              "delete runtime rồi Run all lại trên máy mới (dữ liệu Drive vẫn "
              "nguyên). Tạm thời rơi về đọc thẳng Drive (RẤT chậm).")
if COPY_KEYFRAMES_LOCAL and _kf_ok:
    LOCAL_DATA = Path("/content/data")
    LOCAL_DATA.mkdir(exist_ok=True)
    _ZCACHE = Path("/content/__zip_cache")
    _ZCACHE.mkdir(exist_ok=True)
    _VID_DIR_RE = _re.compile(r"^[A-Z]\d{2}_V\d{3}$")

    def _zip_family(zname: str):
        # CÙNG thứ tự ưu tiên với guess_dest ở ô 5 — một zip phải về đúng
        # MỘT family ở cả hai ô. Videos* trả None: video ở lại Drive (symlink).
        z = zname.lower().replace("_", "-").replace(" ", "-")
        if "map" in z and "keyframe" in z:                        return "map-keyframes"
        if "clip-feature" in z or "features-32" in z:             return "clip-features-32"
        if "media-info" in z or "metadata" in z:                  return "media-info"
        if "object" in z:                                         return "objects"
        if z.startswith(("keyframes", "keyframe", "key-frames")): return "keyframes"
        return None

    def _walk_wrapper(root: Path) -> Path:
        # bỏ các folder bọc ngoài thật sự (Keyframes_L26/keyframes/…) nhưng
        # không bao giờ nhầm một payload dir dạng L21_V001 đơn độc là wrapper
        src = root
        while True:
            ch = list(src.iterdir())
            if len(ch) == 1 and ch[0].is_dir() and not _VID_DIR_RE.match(ch[0].name):
                src = ch[0]
                continue
            return src

    def _merge_into(src: Path, dest: Path) -> int:
        """Move src/* vào dest — không ghi đè, đi sâu 1 cấp cho dir trùng."""
        kept = 0
        dest.mkdir(parents=True, exist_ok=True)
        for item in src.iterdir():
            target = dest / item.name
            if not target.exists():
                shutil.move(str(item), str(target))
            elif item.is_dir() and target.is_dir():
                for sub in item.iterdir():
                    st = target / sub.name
                    if not st.exists():
                        shutil.move(str(sub), str(st))
                    else:
                        kept += 1
            else:
                kept += 1
        return kept

    for _sub in ("keyframes", "map-keyframes", "media-info", "objects", "clip-features-32"):
        dst = LOCAL_DATA / _sub
        _stamp = LOCAL_DATA / f".materialized-{_sub}"
        if _stamp.exists():
            # verify-R14: stamp KHÔNG được che zip mới upload giữa session —
            # còn zip matching chưa có marker local thì phải bung bổ sung.
            _ensure_drive()
            _new = [z for z in sorted(DATA_DIR.glob("*.zip"))
                    if _zip_family(z.name) == _sub
                    and not (LOCAL_DATA / f".unzipped-{_sub}-{z.stem}").exists()]
            if not _new:
                print(f"{_sub}: đã materialize trong session này — skip")
                continue
            print(f"{_sub}: {len(_new)} zip mới sau lần materialize trước → bung bổ sung")
        if dst.exists():
            for stale in dst.glob("*.__tmp"):
                shutil.rmtree(stale, ignore_errors=True) if stale.is_dir() else stale.unlink()

        # PHA 1 — bung từ zip nguồn (marker LOCAL theo từng zip → resume mịn;
        # crash giữa merge không sao: lần sau bung lại, merge chỉ bù file thiếu)
        _ensure_drive()
        for zp in sorted(DATA_DIR.glob("*.zip")):
            if _zip_family(zp.name) != _sub:
                continue
            _done = LOCAL_DATA / f".unzipped-{_sub}-{zp.stem}"
            if _done.exists():
                continue
            t0 = time.time()
            lz = _ZCACHE / zp.name
            tmp_root = _ZCACHE / "__tmp_extract"
            for _attempt in (1, 2, 3):
                try:
                    # verify-R14: MỌI syscall chạm FUSE (stat, copyfile) phải
                    # nằm TRONG retry — zip trước mất nhiều phút extract thuần
                    # local, FUSE có thể chết trong cửa sổ đó.
                    _ensure_drive()
                    _free = shutil.disk_usage("/content").free
                    if _free < zp.stat().st_size * 2.2 + 5e9:
                        raise RuntimeError(          # không retry lỗi hết disk
                            f"Disk local sắp đầy ({_free / 1e9:.0f} GB) — không đủ "
                            f"chỗ bung {zp.name}. Runtime ▸ Disconnect and delete "
                            "runtime để lấy máy mới, hoặc đặt "
                            "COPY_KEYFRAMES_LOCAL=False (chậm hơn nhiều).")
                    shutil.copyfile(zp, lz)             # 1 file to, đọc tuần tự
                    if tmp_root.exists():
                        shutil.rmtree(tmp_root)
                    with zipfile.ZipFile(lz) as z:      # CRC check từng member
                        z.extractall(tmp_root)
                    break
                except (OSError, zipfile.BadZipFile) as e:
                    print(f"   ⚠ {zp.name}: {e!r} — thử lại ({_attempt}/3)")
                    if _attempt == 3:
                        raise
                    time.sleep(5)
            kept = _merge_into(_walk_wrapper(tmp_root), dst)
            shutil.rmtree(tmp_root, ignore_errors=True)
            lz.unlink(missing_ok=True)                  # trả disk ngay
            _done.touch()
            print(f"   {zp.name} → local {_sub}/ ({time.time() - t0:.0f}s"
                  + (f", giữ {kept} mục trùng)" if kept else ")"))

        # PHA 2 — merge phần CHỈ có trên Drive (K-batch tự cắt, upload tay…):
        # 1 lần listdir + exists-check local là rẻ; copy lẻ chỉ cho phần thiếu.
        added = 0
        srcD = DATA_DIR / _sub
        # verify-R14: family chỉ-có-folder (không zip nguồn) → pha 1 chưa hề
        # tạo dst; copy2 vào parent chưa tồn tại sẽ FileNotFoundError.
        dst.mkdir(parents=True, exist_ok=True)
        for _attempt in (1, 2, 3):
            try:
                _ensure_drive()                 # srcD.exists cũng chạm FUSE
                if srcD.exists():
                    for item in sorted(srcD.iterdir()):
                        if item.name.startswith(".unzipped-") or item.name.endswith(".__tmp"):
                            continue
                        target = dst / item.name
                        if target.exists():
                            continue
                        tmp_target = dst / (item.name + ".__tmp")
                        if tmp_target.is_dir():
                            shutil.rmtree(tmp_target)
                        elif tmp_target.exists():
                            tmp_target.unlink()
                        (shutil.copytree if item.is_dir() else shutil.copy2)(item, tmp_target)
                        tmp_target.rename(target)
                        added += 1
                break
            except OSError as e:
                print(f"   ⚠ merge Drive-extras {_sub}: {e!r} — thử lại ({_attempt}/3)")
                if _attempt == 3:
                    raise
                time.sleep(5)
        _stamp.touch()
        _n = sum(1 for _ in dst.iterdir())
        print(f"{_sub}: sẵn sàng local ({_n} mục"
              + (f", +{added} bù từ Drive" if added else "") + ")")
    # INTEGRITY + SELF-HEAL (round-11/12, live-run lessons): Google Drive FUSE
    # can serve freshly-written files back EMPTY (buffered writes lost when a
    # session dies mid-sync). Round-11 hit 873 header-less map csvs; round-12
    # hit empty clip-features .npy files that killed the provided_clip32 lane
    # AFTER 8h of GPU work. Validate every LOCAL small-file artifact and heal
    # broken ones straight FROM THE SOURCE ZIP (uploaded long ago = reliably
    # synced), repairing the Drive copy too.
    import zipfile as _zf
    import numpy as _np

    def _bad_csv(f):
        try:
            if f.stat().st_size < 40:
                return True
            with open(f, encoding="utf-8-sig") as fh:
                return sum(1 for _ in fh) < 2          # header only / empty
        except OSError:
            return True

    def _bad_npy(f):
        try:
            if f.stat().st_size < 90:                  # npy header alone is ~64B
                return True
            return _np.load(f, mmap_mode="r").shape[0] == 0
        except Exception:
            return True

    def _bad_empty(f):
        try:
            return f.stat().st_size == 0
        except OSError:
            return True

    # (subdir, glob, zip-name matcher, validator, key depth 1=basename 2=vid/name)
    _HEAL_SPECS = [
        ("map-keyframes", "*.csv",
         lambda z: "map" in z and "keyframe" in z, _bad_csv, 1),
        ("clip-features-32", "*.npy",
         lambda z: "clip-feature" in z or "features-32" in z, _bad_npy, 1),
        ("media-info", "*.json",
         lambda z: "media-info" in z or "metadata" in z, _bad_empty, 1),
        ("objects", "*/*.json",
         lambda z: "object" in z, _bad_empty, 2),
    ]
    for _sub, _pat, _match, _isbad, _depth in _HEAL_SPECS:
        _dirL = LOCAL_DATA / _sub
        if not _dirL.is_dir():
            continue
        _key = (lambda p: p.name) if _depth == 1 else (lambda p: f"{p.parent.name}/{p.name}")
        _bad = [f for f in sorted(_dirL.glob(_pat)) if _isbad(f)]
        if not _bad:
            print(f"{_sub} integrity: OK")
            continue
        print(f"⚠ {len(_bad)} file LOCAL rỗng/hỏng trong {_sub}/ (Drive FUSE mất "
              "dữ liệu?) — tự phục hồi từ zip gốc ...")
        # Zip handles opened ONCE per family (round-13): re-opening a Drive
        # zip per bad file would stall for hours on a family-scale corruption.
        _ensure_drive()                        # verify-R14: ZipFile đọc qua FUSE
        _members, _open_zips = {}, []
        for _z in DATA_DIR.glob("*.zip"):
            _zl = _z.name.lower().replace("_", "-")
            if _match(_zl):
                _zh = _zf.ZipFile(_z)
                _open_zips.append(_zh)
                for _n in _zh.namelist():
                    if not _n.endswith("/"):
                        _parts = Path(_n).parts
                        _members["/".join(_parts[-_depth:])] = (_zh, _n)
        _healed = 0
        for f in _bad:
            _srcz = _members.get(_key(f))
            if not _srcz:
                continue
            _data = _srcz[0].read(_srcz[1])
            if not _data:
                continue
            f.write_bytes(_data)                       # heal LOCAL
            _drv = DATA_DIR / _sub / _key(f)           # heal DRIVE too
            try:
                if not _drv.exists() or _isbad(_drv):
                    _tmpf = _drv.parent / (_drv.name + ".__tmp")
                    _tmpf.write_bytes(_data)
                    _tmpf.replace(_drv)
            except OSError:
                pass
            _healed += 1
        for _zh in _open_zips:
            _zh.close()
        print(f"   phục hồi {_healed}/{len(_bad)}")
        _still = [_key(f) for f in _bad if _isbad(f)]
        if _still:
            raise RuntimeError(
                f"{len(_still)} file trong {_sub}/ vẫn hỏng sau phục hồi "
                f"(vd {_still[:3]}) — kiểm tra zip nguồn còn trong data/ trên "
                "Drive (đừng xóa zip!) rồi chạy lại ô này.")
    # videos stay on Drive (huge); link them in
    _ensure_drive()
    if (DATA_DIR / "videos").exists() and not (LOCAL_DATA / "videos").exists():
        (LOCAL_DATA / "videos").symlink_to(DATA_DIR / "videos")
    import os
    os.environ["CVP_PATHS__DATA_ROOT"] = str(LOCAL_DATA)
    from cvp.config import load_settings
    settings = load_settings()
    print("data_root now:", settings.paths.data_root)
else:
    print("using Drive data_root directly")

In [ ]:
# ── 5b · Compact objects index (một lần, nếu nb01 chưa build) ──
# verify-R23 (HIGH): thiếu objects.parquet thì ObjectBooster rơi về đọc từng
# file json qua Drive FUSE — cộng thêm HÀNG PHÚT mỗi query. Build từ đĩa
# local (ô 5 đã materialize) rồi ghi MỘT file parquet lên Drive artifacts.
from cvp.config import load_settings
from cvp.data.catalog import KeyframeCatalog
from cvp.data.objects_compact import build_objects_index

import time as _t
from pathlib import Path as _P

settings = load_settings()
_pq = settings.paths.art("objects_index") / "objects.parquet"
# round-28: stat lười trên VM mới từng nói parquet "không tồn tại" dù nó nằm
# sẵn trên Drive → suýt rebuild vô ích (và chết nếu data_root chưa local).
# Nudge-poll trước khi kết luận vắng mặt.
for _w in range(6):
    try:
        list(_P(str(settings.paths.artifacts_root)).iterdir())   # nudge metadata
    except OSError:
        pass
    if _pq.exists():
        break
    _t.sleep(5)
if _pq.exists():
    print("objects.parquet: đã có —", _pq)
else:
    catalog = KeyframeCatalog(settings)
    catalog.build()
    print("objects.parquet built:", build_objects_index(settings, catalog))

In [ ]:
# ── 5c · Artifacts đọc-nhiều → đĩa LOCAL (round-26, live nb03 run 2) ──
# Engine mmap embeddings/index từ Drive FUSE → query "lạnh" 114-130s, query
# "ấm" 1.75s. Copy các thư mục CHỈ-ĐỌC về local (~4-6GB, vài phút) rồi trỏ
# artifacts_root vào đó. Drive KHÔNG bị đụng — submissions được đồng bộ ngược
# về Drive ở ô đóng gói.
import os, shutil, time
from pathlib import Path

LOCAL_ART = Path("/content/artifacts")
LOCAL_ART.mkdir(exist_ok=True)
_READ_HOT = ("catalog", "embeddings", "indexes", "text_index", "objects_index",
             "asr", "checkpoints", "thumbs")   # round-42: webp thumbs → lưới UI 10x
_t0 = time.time()
try:
    list(ARTIFACTS.iterdir())    # round-28: nudge metadata trước loạt exists()
except OSError:
    _ensure_drive()
_missing = []
for _d in _READ_HOT:
    src, dst = ARTIFACTS / _d, LOCAL_ART / _d
    if dst.exists():
        print(f"   {_d}/: đã có local — skip")
        continue
    _ensure_drive()
    if not src.exists():
        _missing.append(_d)          # round-76: thiếu là phải LA LÊN, xem dưới
        continue
    _tmp = LOCAL_ART / (_d + ".__tmp")
    if _tmp.exists():
        shutil.rmtree(_tmp)
    shutil.copytree(src, _tmp)
    _tmp.rename(dst)
    print(f"   {_d}/ → local")
if _missing:
    # Round-76 (audit tiền-trận): trước đây thiếu thư mục nào là LẶNG LẼ bỏ
    # qua — text_index vắng mặt nghĩa là OCR/ASR/caption âm thầm = 0 suốt
    # trận. Metadata DriveFS lười là thủ phạm quen mặt; thuốc: đổi máy ảo.
    print(f"\n⚠⚠⚠ THIẾU {len(_missing)} kho artifacts trên Drive: {_missing}")
    print("    Máy ảo lười metadata? ĐỔI MÁY ẢO MỚI rồi Run all lại —")
    print("    KHÔNG ra trận khi thiếu bất kỳ kho nào ngoài 'thumbs'.")
(LOCAL_ART / "submissions").mkdir(parents=True, exist_ok=True)
os.environ["CVP_PATHS__ARTIFACTS_ROOT"] = str(LOCAL_ART)
# round-42: có kho thumbnail (chạy scripts/60_make_thumbs.py MỘT lần) → web
# đội tải ảnh ~8KB thay vì 60-150KB — lưới hiện gần như tức thì qua tunnel.
if (LOCAL_ART / "thumbs").is_dir():
    os.environ["CVP_WEB__THUMBS_DIR"] = str(LOCAL_ART / "thumbs")
    print("thumbs: BẬT (webp 320px)")
print(f"artifacts_root now: {LOCAL_ART} ({time.time() - _t0:.0f}s)")

In [ ]:
# ── 5 · Load the search engine ──
# Chọn model cho phiên test này:
#   "siglip2"    lane gốc zero-shot (dự phòng)
#   "finetuned"  text tower tiếng Việt từ nb02 (cùng index ảnh siglip2)
#   "ensemble"   round-47: finetuned + METACLIP2 60/40 — bench Lab 23/08:
#                0.5522 vs finetuned đơn 0.5370 (retrieval-thuần, 23/23 câu).
#                (Cặp cũ finetuned+openclip đã thua A/B 20/08 và bị thay.)
ENGINE_MODEL   = "ensemble"         # ← cấu hình ra trận đợt 2 (bench 23/08)
# "none"   = test offline, không gọi Gemini (nhanh, không tốn quota)
# "gemini" = dịch + mở rộng query (CẦN secret GEMINI_API_KEY; tự rơi về
#            Google-Translate miễn phí rồi passthrough nếu API lỗi — A/B 20/08:
#            riêng bản dịch EN đã nâng chất lượng rõ rệt cho mọi lane)
QUERY_PROVIDER = "gemini"           # ← cấu hình ra trận round 1
# Gemini NHÌN top-24 ảnh ứng viên và xếp lại đầu bảng (UIT CVPRW'25: +10%
# hit@1). Cần API trả phí; ~3–8s/query. Tắt (False) nếu cần UI phản hồi nhanh.
VLM_RERANK = True
# Khẩu pháo cuối: cross-encoder Qwen3-VL-Reranker-2B chạy LOCAL trên A100,
# chấm lại từng cặp (câu, ảnh) trong top-100 rồi trộn 50/50 với điểm fusion
# (recipe Unified-IMMR AIC-2025). Bổ trợ cho VLM rerank (pairwise ↔ listwise);
# mọi đường lỗi tự trả về thứ hạng cũ. ĐÃ ĐO ở vòng nháp 20/08: 7.2 → 7.6
# (VLM rerank trước đó: 6.4 → 7.2) — giữ True cho round 1.
CROSS_RERANK = True
# Round-37, học từ bài 19.8/23 (vòng nháp): đáp án chuẩn hay đứng rank 25–79
# trong bảng của ta — NGOÀI tầm nhìn top-24 của VLM rerank. Nới lên 48 để
# Gemini với tới (vẫn MỘT cuộc gọi, chỉ nhiều ảnh hơn, thêm ~2–4s/query).
VLM_RERANK_TOPK = 48
# Round-40 "suy nghĩ lâu hơn": gọi Gemini nhiều lần và biểu quyết. Bằng chứng
# trận 21/08: cùng cấu hình ra 9.4 rồi 9.0 (xúc xắc VLM); QA q3 lật '300 kg'
# ↔ '30 kg' giữa hai lần chạy. 3 phiếu đổi ~2× thời gian pack lấy độ ổn định.
VLM_VOTES = 3      # VLM rerank: trung bình 3 lượt chấm (1 = tắt)
QA_VOTES  = 3      # VQA: 3 lần trả lời, lấy đáp án đa số (1 = tắt)
import os, time
from pathlib import Path
os.environ["CVP_EMBEDDING__MODEL"] = ENGINE_MODEL
os.environ["CVP_QUERY__PROVIDER"]  = QUERY_PROVIDER
os.environ["CVP_SEARCH__VLM_RERANK"] = "true" if VLM_RERANK else "false"
os.environ["CVP_SEARCH__VLM_RERANK_TOPK"] = str(VLM_RERANK_TOPK)
os.environ["CVP_SEARCH__VLM_RERANK_VOTES"] = str(VLM_VOTES)
os.environ["CVP_VQA__SELF_CONSISTENCY"] = str(QA_VOTES)
os.environ["CVP_SEARCH__RERANKER"] = "qwen_reranker" if CROSS_RERANK else "none"
# Round-44: Lab (nb04) dò được bộ trọng số fusion thắng bench → tự nạp.
_tw = PROJECT / "artifacts" / "tuning" / "best_weights.json"
try:                                     # round-76: nudge metadata tuning/
    list(_tw.parent.iterdir())           # (nb04 có từ round-71, nb03 thì chưa)
except OSError:
    pass
if not _tw.exists():
    print("⚠⚠ KHÔNG thấy tuning/best_weights.json — trận sẽ chạy trọng số "
          "MẶC ĐỊNH, KHÁC cấu hình bench 0.6826! Metadata Drive lười? "
          "Chạy lại cell này; vẫn thiếu thì đổi máy ảo mới.")
if _tw.exists():
    import json as _json
    _w = _json.loads(_tw.read_text(encoding="utf-8")).get("best", {}).get("weights")
    if _w:
        # Round-49 SHRINKAGE 50% về baseline: tuner học trên vỏn vẹn 23 câu
        # của pack NHÁP và kéo OCR về ≈0.01 — đề đợt sau phân bố khác, tin
        # 100% cực trị đó là đánh bạc overfit. Trung bình với baseline giữ
        # nguyên CHIỀU HƯỚNG đã học (hạ OCR/ASR, nâng caption/metadata)
        # nhưng chỉ đi nửa biên độ — không tín hiệu nào bị giết hẳn.
        _base = {"visual": 1.0, "ocr": 0.35, "asr": 0.30, "caption": 0.25,
                 "metadata": 0.15, "object": 0.25}
        _w = {k: round(0.5 * float(_w.get(k, v)) + 0.5 * v, 4)
              for k, v in _base.items()}
        for _sig, _val in _w.items():
            os.environ[f"CVP_SEARCH__WEIGHTS__{_sig.upper()}"] = str(_val)
        print("⚖ Trọng số fusion TUNED+shrinkage 50% (từ Lab):", _w)
# Ranking phẳng (top không tách khỏi đám đông) → tự tìm lại bằng các biến thể
# Gemini đã cache rồi trộn RRF — không tốn thêm cuộc gọi API nào.
os.environ["CVP_SEARCH__LOW_CONFIDENCE_RETRY"] = "true"
# ── Round-75: gói knob AB vào TRẬN — thắng bench#3/#4 (0.6478 → 0.6739 →
# 0.6826; hiệu ứng nhất quán 2 lượt: q22-qa +0.8, TRAKE +0.2, đuôi KIS +0.4,
# đổi q24 −0.2). Gói B (3 knob vqa) không đo được lãi trên đề nháp nhưng là
# bảo hiểm định dạng cho đề thật, không phá gì (q22 giữ 0.8, budget đủ).
for _k, _v in {
    "CVP_SEARCH__NEIGHBOR_CONSISTENCY_BOOST": "0.15",
    "CVP_SEARCH__ROW_STRATEGY": "diversify_tail",
    "CVP_TEMPORAL__SUBMIT_STRATEGY": "jitter",
    "CVP_TEMPORAL__POOL_CONTEXT": "prepend",
    "CVP_TEMPORAL__EVENT_QUERY_VARIANTS": "all",
    "CVP_TEMPORAL__CAPTION_SIGNAL_WEIGHT": "0.2",
    "CVP_VQA__ANSWER_CANONICALIZE": "true",
    "CVP_VQA__ANSWER_NEIGHBOR_FRAMES": "1",
    "CVP_VQA__MAX_CALLS_PER_QUERY": "10",
}.items():
    os.environ[_k] = _v
print("🎛 Gói knob AB (round-75) đã vào trận: boost 0.15 + diversify_tail + "
      "4 knob TRAKE + vote canonical/neighbor")
if ENGINE_MODEL in ("finetuned", "ensemble"):
    os.environ["CVP_FINETUNED__CHECKPOINT"] = str(
        Path(os.environ["CVP_PATHS__ARTIFACTS_ROOT"]) / "checkpoints" / "vi_siglip2_best")
if ENGINE_MODEL == "ensemble":
    os.environ["CVP_EMBEDDING__ENSEMBLE_MEMBERS"] = '["finetuned", "metaclip2"]'
    os.environ["CVP_EMBEDDING__ENSEMBLE_WEIGHTS"] = "[0.6, 0.4]"

# Round-76 (audit tiền-trận): thiếu GEMINI key là hỏng ÂM THẦM và MUỘN —
# enhancement rơi về Google-Translate, VLM rerank tắt, QA rơi về Vintern;
# cấu hình 0.6826 cần Gemini. La lên NGAY tại đây thay vì giữa trận.
if QUERY_PROVIDER == "gemini" and not (os.environ.get("GEMINI_API_KEY")
                                       or os.environ.get("GOOGLE_API_KEY")):
    print("⚠⚠⚠ KHÔNG có GEMINI_API_KEY/GOOGLE_API_KEY trong env — thêm secret "
          "ở panel 🔑 (bật Notebook access), chạy lại cell secrets rồi cell "
          "này. Ra trận thiếu key = mất enhancement + VLM rerank + QA Pro!")
from cvp.search.engine import SearchEngine
from cvp.config import load_settings
settings = load_settings()
t0 = time.time()
engine = SearchEngine(settings)
# Round-76: chốt chặn 2-lane cho TRẬN — nb04 có từ round-71, nb03 thì chưa.
# Engine "degrade gracefully" khi một lane hỏng: đêm thi mà chạy ensemble
# thiếu lane là đánh cả đêm với nửa vũ khí, chỉ có một dòng warning chìm
# trong log. Fail TO TIẾNG tại đây; thuốc: đổi máy ảo MỚI rồi Run all lại.
if ENGINE_MODEL == "ensemble":
    assert getattr(engine, "member_names", None) == ["finetuned", "metaclip2"], (
        f"Ensemble thiếu lane: {getattr(engine, 'member_names', None)} — "
        "index/checkpoint chưa nạp đủ (máy ảo lười metadata?). Đổi máy ảo "
        "MỚI và Run all lại — TUYỆT ĐỐI không ra trận thiếu lane.")
    print("✓ ensemble đủ 2 lane:", engine.member_names)
print(f"engine ready in {time.time()-t0:.1f}s — {len(engine.catalog):,} keyframes")

In [ ]:
# ── 6 · Run sample Vietnamese KIS queries + show results inline ──
import time
import matplotlib.pyplot as plt
from PIL import Image

QUERIES = [
    "người dẫn chương trình mặc áo dài đứng trong trường quay",
    "đám cháy lớn khói đen bốc lên từ tòa nhà",
    "các vận động viên đua xe đạp trên đường phố",
    "món ăn được trình bày trên đĩa trắng",
    "cảnh ngập lụt trên đường phố, người dân lội nước",
]

for q in QUERIES:
    t0 = time.time()
    results = engine.search_text(q, display_k=8)
    dt = time.time() - t0
    print(f"\n🔎 {q}   ({dt:.2f}s)")
    if not results:
        print("   (no results)")
        continue
    fig, axes = plt.subplots(1, min(8, len(results)), figsize=(20, 3), squeeze=False)
    for ax, r in zip(axes[0], results):
        try:
            ax.imshow(Image.open(r.ref.path)); ax.axis("off")
            ax.set_title(f"{r.video_id}\nf={r.frame_idx} s={r.score:.2f}", fontsize=7)
        except Exception:
            ax.axis("off")
    plt.show()

In [ ]:
# ── 7 · TRAKE + AVS smoke test ──
events = [
    "vận động viên chuẩn bị xuất phát",
    "vận động viên chạy trên đường đua",
    "vận động viên về đích ăn mừng",
]
seqs = engine.search_trake(events, max_results=5)
for c in seqs[:5]:
    print(f"TRAKE {c.video_id} frames={c.frame_idxs} score={c.score:.3f}")

avs = engine.search_avs("cảnh giao thông đông đúc ở thành phố", limit=20)
print(f"\nAVS: {len(avs)} rows across {len({r.video_id for r in avs})} distinct videos")

In [ ]:
# ── 8 · Submission CSV round-trip (exact Codabench format) ──
from cvp.submission.writer import write_kis, write_trake

results = engine.search_text(QUERIES[0])
p = write_kis(settings.paths.art("submissions", "demo-kis.csv"),
              [(r.video_id, r.frame_idx) for r in results])
print(p, "\n" + p.read_text(encoding="utf-8")[:300])

nb03_files = [p]   # ONLY the CSVs this notebook run writes get packaged (cell 9)
if seqs:
    p2 = write_trake(settings.paths.art("submissions", "demo-trake.csv"),
                     [(c.video_id, c.frame_idxs) for c in seqs])
    nb03_files.append(p2)
    print(p2, "\n" + p2.read_text(encoding="utf-8")[:300])

In [ ]:
# ── 9 · Validate + package (Codabench) ──
# The organiser contract is re-checked on the finished CSVs (a wrong row
# silently costs a submission slot); errors BLOCK the zip. Validate and zip
# ONLY the CSVs cell 8 just wrote (nb03_files) — artifacts/submissions is
# shared with the UI's real exports, and stale/demo files must never mix
# into a contest zip (packager docstring contract).
import json
from cvp.submission.packager import has_errors, package_codabench, validate_file

sub_dir = settings.paths.art("submissions")
issues = [i for f in nb03_files for i in validate_file(f, strict=True)]
for i in issues:
    print(" ", i)
if has_errors(issues):
    print("❌ Còn lỗi chặn — sửa CSV rồi chạy lại ô này (zip KHÔNG được tạo).")
else:
    zip_path = settings.paths.art("submissions", "submission.zip")
    package_codabench(sub_dir, zip_path, package_name=settings.submission.package_name,
                      files=nb03_files)
    manifest = json.loads((zip_path.parent / "MANIFEST.json").read_text(encoding="utf-8"))
    print(f"\n📦 {zip_path.name}  sha256={manifest['zip_sha256'][:12]}…")
    for f in manifest["files"]:
        print(f"   {f['name']:24} task={f['task']:5} rows={f['rows']}")
    # round-26: artifacts_root đang là LOCAL — đồng bộ submissions về Drive để
    # zip/csv sống sót sau khi phiên tắt (chỉ THÊM/GHI ĐÈ file cùng tên của
    # chính lượt chạy này, không xóa gì trên Drive).
    import shutil as _sh
    _drv_sub = ARTIFACTS / "submissions"
    if Path(str(sub_dir)).resolve() != _drv_sub.resolve():
        _ensure_drive()
        _sh.copytree(sub_dir, _drv_sub, dirs_exist_ok=True)
        print("submissions đồng bộ về Drive:", _drv_sub)

In [ ]:
# ── 9b · 🏁 THI ĐẤU THẬT: chạy CẢ PACK đề của BTC → CSV → validate → zip ──
# Chuẩn bị: bung zip đề của BTC, upload các file query-*.txt vào
# MyDrive/<project>/queries/<QUERY_PACK>/  (file .txt nằm TRỰC TIẾP trong
# thư mục — đừng để lồng thêm một thư mục con sau khi bung zip).
QUERY_PACK = "p1"      # tên thư mục con trong queries/
RUN_PACK   = False     # bật True khi đề đã nằm đúng chỗ
REZIP_ONLY = False     # True: KHÔNG search lại — chỉ validate + zip lại các
                       # query-*.csv hiện có trong pack (dùng SAU khi soát tay
                       # bằng UI và ghi đè vài file CSV bằng bản người chọn)
_qdir = PROJECT / "queries" / QUERY_PACK
if RUN_PACK and not (_qdir.is_dir() and any(_qdir.glob("*.txt"))):
    # Round-34: đêm thi Run all chạy TRƯỚC giờ BTC phát đề — cell này crash
    # là đứt Run all và cell UI phía dưới không bao giờ mở. Báo rồi đi tiếp.
    print(f"⚠ Chưa thấy file query-*.txt trong {_qdir} — upload đề vào đó rồi "
          "chạy lại RIÊNG cell này. (Run all vẫn đi tiếp, engine + UI không bị chặn.)")
elif RUN_PACK:
    import shutil as _sh

    _out = settings.paths.art("submissions", QUERY_PACK)
    if REZIP_ONLY:
        from cvp.submission.packager import has_errors, package_codabench, validate_file

        class rep:  # noqa: N801 — cùng hình dạng với AutoRunReport phía dưới
            written = sorted(_out.glob("query-*.csv"))
            failed: dict = {}
            issues = [i for p in written for i in validate_file(p, strict=True)]
            zip_path = None
        if rep.written and not has_errors(rep.issues):
            _zp = _out / f"{settings.submission.package_name}.zip"
            if not has_errors(package_codabench(
                    _out, _zp, package_name=settings.submission.package_name,
                    files=rep.written)):
                rep.zip_path = _zp
    else:
        from cvp.pipeline.auto_agent import run_auto

        # engine_factory tái dùng engine ô 5 (đang nóng, đúng cấu hình ra
        # trận) — để mặc định sẽ build engine THỨ HAI và nhân đôi RAM/VRAM.
        rep = run_auto(_qdir, _out, settings, submit=False,
                       engine_factory=lambda _s: engine)
    print(f"\nCSV viết được: {len(rep.written)}")
    if rep.failed:
        print(f"⚠ {len(rep.failed)} query KHÔNG ra CSV (sẽ 0 điểm): {sorted(rep.failed)}")
    for _i in rep.issues:
        print("  ", _i)
    if rep.zip_path:
        _drv = PROJECT / "artifacts" / "submissions" / QUERY_PACK
        _drv.mkdir(parents=True, exist_ok=True)
        _sh.copytree(_out, _drv, dirs_exist_ok=True)
        print(f"\n📦 {rep.zip_path.name} đã đồng bộ về Drive: {_drv}")
        print("→ Tải submission.zip từ Drive về máy, nộp ở tab 'Nộp bài' của BTC.")
    else:
        print("⚠ KHÔNG có zip — sửa lỗi validate ở trên rồi chạy lại cell này "
              "(đừng nộp tay CSV lẻ).")
else:
    print("RUN_PACK=False — upload đề vào queries/<PACK>/ rồi bật True và chạy lại.")

In [ ]:
# ── 10 · (optional) Score against ground truth — official formulas ──
# Drop a GT file at MyDrive/<project>/queries/gt.json to see the exact
# Codabench-style scores (R@k / Final) of the CSVs written above.
GT_PATH = PROJECT / "queries" / "gt.json"
if GT_PATH.exists():
    from cvp.eval.official import score_run
    report = score_run(settings.paths.art("submissions"), GT_PATH)
    print(f"{'query':28} {'task':6} {'final':>7} {'best_rank':>9}")
    for stem, qs in sorted(report.per_query.items()):
        print(f"{stem:28} {qs.task:6} {qs.final:>7.4f} {str(qs.best_rank):>9}")
    for stem, why in sorted(report.unscored.items()):
        print(f"{stem:28} UNSCORED: {why}")
    print(f"\nmean_final = {report.mean_final:.4f}  "
          f"({report.num_scored}/{report.num_gt} GT queries scored)")
else:
    print("Không thấy", GT_PATH, "— muốn chấm điểm offline, tạo JSON keyed theo")
    print("tên file CSV (không đuôi). Ví dụ tối thiểu:")
    print("""{
  "demo-kis": {"task": "kis", "video_id": "L21_V001", "range": [500, 510]}
}""")
    print("Hỗ trợ range / center+epsilon / moments / answers / targets (AVS coverage) "
          "— xem cvp/eval/official.py")

In [ ]:
# ── 11 · (optional) Automatic track dry-run (no human, no submit) ──
# Runs the full auto pipeline over MyDrive/<project>/queries/example/*.txt:
# infer task per filename → search → CSVs → validate → Codabench zip.
RUN_AUTO_AGENT = False
if RUN_AUTO_AGENT:
    from cvp.pipeline.auto_agent import run_auto
    qdir = PROJECT / "queries" / "example"
    qdir.mkdir(parents=True, exist_ok=True)
    if not any(qdir.glob("*.txt")):     # seed one sample query for the dry-run
        (qdir / "query-p1-1-kis.txt").write_text(
            "người dẫn chương trình mặc áo dài đứng trong trường quay", encoding="utf-8")
    # Reuse the cell-5 engine (review C6): the default engine_factory would
    # build a SECOND full engine and double index+catalog memory this session.
    rep = run_auto(qdir, settings.paths.art("submissions", "auto_dry_run"),
                   settings, submit=False, engine_factory=lambda _s: engine)
    print(f"AutoRunReport: written={len(rep.written)} ok={rep.ok} zip={rep.zip_path}")
    for i in rep.issues:
        print("  ", i)
else:
    print("RUN_AUTO_AGENT=False — bật để chạy thử automatic track (submit=False).")

In [ ]:
# ── 12 · (optional) Launch the Streamlit UI from Colab ──
# Colab can't open localhost — use the built-in proxy:
LAUNCH_UI = False
SHARE_URL = True    # round-32: link tạm cho ĐỒNG ĐỘI cùng vào UI (cửa sổ
                    # proxy của Colab CHỈ chủ phiên xem được). Link chỉ mở
                    # giao diện tìm kiếm — KHÔNG lộ source/Drive/notebook.
                    # Chỉ gửi link trong nhóm kín; tắt phiên là link chết.
if LAUNCH_UI:
    import os, re, subprocess, time
    # round-30 ĐẢO NGƯỢC verify-R23: ô engine giờ đặt sẵn CẤU HÌNH RA TRẬN
    # (finetuned + gemini + artifacts local) — UI PHẢI thừa hưởng env này;
    # pop như trước sẽ âm thầm hạ UI về siglip2 zero-shot đọc Drive.
    _env = dict(os.environ)
    proc = subprocess.Popen(
        ["streamlit", "run", str(REPO_DIR / "app" / "streamlit_app.py"),
         "--server.port", "8501", "--server.headless", "true"], env=_env)
    time.sleep(8)
    if SHARE_URL:
        # cloudflared quick tunnel: không cần tài khoản, URL ngẫu nhiên dài
        # khó đoán, tự chết khi phiên tắt.
        _cf = "/content/cloudflared"
        if not os.path.exists(_cf):
            subprocess.run(
                ["wget", "-q", "-O", _cf,
                 "https://github.com/cloudflare/cloudflared/releases/"
                 "latest/download/cloudflared-linux-amd64"], check=True)
            os.chmod(_cf, 0o755)
        _tun = subprocess.Popen([_cf, "tunnel", "--url", "http://localhost:8501"],
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True)
        _url, _t0 = None, time.time()
        while _url is None and time.time() - _t0 < 90:
            _m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com",
                           _tun.stdout.readline() or "")
            if _m:
                _url = _m.group(0)
        if _url:
            print("🔗 LINK CHO ĐỒNG ĐỘI (gửi trong nhóm kín):", _url)
        else:
            print("⚠ Không lấy được URL tunnel sau 90s — chạy lại cell, "
                  "hoặc dùng cửa sổ proxy bên dưới (chỉ mình bạn xem được).")
    from google.colab import output
    output.serve_kernel_port_as_window(8501)   # cửa sổ riêng của CHỦ PHIÊN
    # proc.terminate() when done
else:
    print("Set LAUNCH_UI=True to serve the app (better: run it on your laptop).")

In [ ]:
# ── 12b · ⚡ Web soát tay NHANH (tùy chọn 2 — SPA + API, không lag rerun) ──
# Chạy NGAY TRONG kernel này, tái dùng engine ô 5 (không tốn thêm VRAM), phát
# qua tunnel thứ hai. Cả đội đăng nhập MỘT tài khoản chung — ai không có
# mật khẩu thì link cũng vô dụng. Streamlit (ô 12) vẫn là phương án dự phòng.
LAUNCH_FAST_UI = True
TEAM_USER = "aic2026-222"
TEAM_PASS = ""            # để trống = tự sinh mật khẩu ngẫu nhiên và in ra
if LAUNCH_FAST_UI:
    import os, re, secrets, subprocess, sys, threading, time
    try:
        import fastapi, uvicorn  # noqa: F401
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "fastapi", "uvicorn"], check=True)
        import uvicorn  # noqa: F401
    import uvicorn

    from cvp.web.server import create_team_app

    if not TEAM_PASS:
        TEAM_PASS = secrets.token_urlsafe(6)
    _webapp = create_team_app(engine, settings, TEAM_USER, TEAM_PASS)
    threading.Thread(
        target=lambda: uvicorn.run(_webapp, host="0.0.0.0", port=8600,
                                   log_level="warning"),
        daemon=True, name="cvp-fast-ui").start()
    time.sleep(3)
    _cf = "/content/cloudflared"
    if not os.path.exists(_cf):
        subprocess.run(["wget", "-q", "-O", _cf,
                        "https://github.com/cloudflare/cloudflared/releases/"
                        "latest/download/cloudflared-linux-amd64"], check=True)
        os.chmod(_cf, 0o755)
    _tun2 = subprocess.Popen([_cf, "tunnel", "--url", "http://localhost:8600"],
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True)
    _u2, _t2 = None, time.time()
    while _u2 is None and time.time() - _t2 < 90:
        _m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com",
                       _tun2.stdout.readline() or "")
        if _m:
            _u2 = _m.group(0)
    if _u2:
        print("⚡ LINK WEB NHANH CHO ĐỒNG ĐỘI:", _u2)
        print(f"   Đăng nhập MỘT tài khoản chung: {TEAM_USER} / {TEAM_PASS}")
        print("   (gửi cả link lẫn mật khẩu trong nhóm kín)")
    else:
        print("⚠ Không lấy được URL tunnel web nhanh — chạy lại cell, "
              "hoặc cả đội dùng link Streamlit ô 12.")
else:
    print("LAUNCH_FAST_UI=False — đội chỉ dùng Streamlit (ô 12).")

In [ ]:
# ── 13 · 🫀 Giữ phiên + trông UI (chạy MÃI — bấm nút Dừng ⏹ của ô để thoát) ──
# Colab thu hồi máy ảo "không hoạt động". Kernel đang bận chạy ô này = hoạt
# động, nên phiên sống tới khi bạn tự dừng (trần cứng 24h của Colab vẫn áp
# dụng). Kiêm watchdog: streamlit / tunnel chết là tự dựng lại và in link MỚI
# (round-35 — phiên nháp 21/08 tự ngắt giữa buổi soát tay, đứt UI cả đội).
KEEP_ALIVE = True
if KEEP_ALIVE:
    import os as _os, re as _re, subprocess as _sp, time as _tm

    def _alive(p):
        return p is not None and p.poll() is None

    _n = 0
    print("🫀 Watchdog chạy — phiên được giữ sống. Bấm nút Dừng (⏹) của ô này "
          "khi muốn kết thúc.")
    try:
        while True:
            if globals().get("LAUNCH_UI"):
                if not _alive(globals().get("proc")):
                    print(f"⚠ {_tm.strftime('%H:%M')} streamlit chết — dựng lại ...")
                    proc = _sp.Popen(
                        ["streamlit", "run", str(REPO_DIR / "app" / "streamlit_app.py"),
                         "--server.port", "8501", "--server.headless", "true"],
                        env=dict(_os.environ))
                    _tm.sleep(8)
                if (globals().get("SHARE_URL") and globals().get("_cf")
                        and not _alive(globals().get("_tun"))):
                    print(f"⚠ {_tm.strftime('%H:%M')} tunnel Streamlit chết — mở lại ...")
                    _tun = _sp.Popen([_cf, "tunnel", "--url", "http://localhost:8501"],
                                     stdout=_sp.PIPE, stderr=_sp.STDOUT, text=True)
                    _u, _t1 = None, _tm.time()
                    while _u is None and _tm.time() - _t1 < 90:
                        _m2 = _re.search(r"https://[a-z0-9-]+\.trycloudflare\.com",
                                         _tun.stdout.readline() or "")
                        if _m2:
                            _u = _m2.group(0)
                    print("🔗 LINK MỚI CHO ĐỒNG ĐỘI:", _u or "⚠ không lấy được — chạy lại ô UI")
            if (globals().get("LAUNCH_FAST_UI") and globals().get("_cf")
                    and globals().get("_tun2") is not None
                    and not _alive(globals().get("_tun2"))):
                # round-36: web nhanh (ô 12b) cũng được watchdog trông hộ
                print(f"⚠ {_tm.strftime('%H:%M')} tunnel web nhanh chết — mở lại ...")
                _tun2 = _sp.Popen([_cf, "tunnel", "--url", "http://localhost:8600"],
                                  stdout=_sp.PIPE, stderr=_sp.STDOUT, text=True)
                _u2, _t2 = None, _tm.time()
                while _u2 is None and _tm.time() - _t2 < 90:
                    _m3 = _re.search(r"https://[a-z0-9-]+\.trycloudflare\.com",
                                     _tun2.stdout.readline() or "")
                    if _m3:
                        _u2 = _m3.group(0)
                print("⚡ LINK WEB NHANH MỚI:", _u2 or "⚠ không lấy được — chạy lại ô 12b")
            _tm.sleep(30)
            _n += 1
            if _n % 10 == 0:   # ~5 phút một nhịp, giữ output gọn
                print(f"🫀 {_tm.strftime('%H:%M')} phiên sống"
                      + (" · UI ok" if _alive(globals().get("proc")) else ""))
    except KeyboardInterrupt:
        print("⏹ Đã dừng watchdog — từ giờ Colab tính phiên là nhàn rỗi.")
else:
    print("KEEP_ALIVE=False — phiên sẽ tự ngắt khi nhàn rỗi.")